# LF6 bridge — manifest suy giảm → schema votes chung

`lf6_degradation` ghi `labels/lf6_degradation_manifest.csv` (base_image, axis, delta, delta_star, vote) — KHÔNG theo schema votes.
Notebook này chuyển sang `labels/votes/lf6_degradation.csv` (lf, image_id, task, vote, path, + axis/delta) để fusion gộp và train_iqa tái tạo ảnh suy giảm.
`image_id = base_image__axis__delta`; `path` lấy lại từ phiếu LF1–5 (ảnh gốc). Xem `docs/LF6_Methodology.md`.

## Cấu hình + module dùng chung

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RUNNER = 'local'

def build_root(runner):
    if runner == 'local':
        for cand in [Path.cwd(), Path.cwd().parent]:
            if (cand / 'labels' / 'lf6_degradation_manifest.csv').exists():
                return cand
        raise SystemExit('Khong thay labels/lf6_degradation_manifest.csv — chay/commit LF6 truoc')
    if runner == 'kaggle':
        root = Path('/kaggle/input/coconut-iqa')
        if not (root / 'labels' / 'lf6_degradation_manifest.csv').exists():
            raise SystemExit('Khong thay manifest tren kaggle')
        return root
    raise SystemExit("RUNNER phai la 'local' hoac 'kaggle'")

ROOT = build_root(RUNNER)
MANIFEST = ROOT / 'labels' / 'lf6_degradation_manifest.csv'
VOTES_DIR = ROOT / 'labels' / 'votes'
OUT = VOTES_DIR / 'lf6_degradation.csv'
UTILS = ROOT / 'src' / 'utils' / 'lf_io.ipynb'
if not UTILS.exists():
    raise SystemExit('Khong thay ' + str(UTILS))
get_ipython().run_line_magic('run', str(UTILS))
print('MANIFEST:', MANIFEST)
print('OUT:', OUT)

## 1. Map base_image → path (ảnh gốc) từ phiếu LF1–5

In [ ]:
def base_paths(votes_dir):
    mapping = {}
    for f in sorted(votes_dir.glob('lf*.csv')):
        if f.name == 'lf6_degradation.csv':
            continue
        d = pd.read_csv(f)
        if 'image_id' not in d.columns or 'path' not in d.columns:
            continue
        for r in d.itertuples():
            if isinstance(r.path, str):
                mapping[str(r.image_id)] = r.path
    return mapping

ID2PATH = base_paths(VOTES_DIR)
print('base_image co path:', len(ID2PATH))

## 2. Chuyển manifest → phiếu (schema chung)

In [ ]:
man = pd.read_csv(MANIFEST)
print('manifest dong:', len(man))

missing = 0
rows = []
for r in man.itertuples():
    base = str(r.base_image)
    path = ID2PATH.get(base, '')
    if path == '':
        missing = missing + 1
    synth_id = base + '__' + str(r.axis) + '__' + str(r.delta)
    rows.append(make_vote(
        lf='lf6_degradation',
        image_id=synth_id,
        task=r.task,
        vote=r.vote,
        source=str(r.source),
        path=path,
        base_image=base,
        original_id=str(r.original_id),
        fold=r.fold,
        axis=str(r.axis),
        delta=r.delta,
        delta_star=r.delta_star,
    ))

print('base_image thieu path:', missing)
write_lf_votes(OUT, rows, extra_fields=['base_image', 'original_id', 'fold', 'axis', 'delta', 'delta_star'])

Kiểm định LF6 (đường cong suy giảm) ở notebook riêng — không nhét vào đây.